In [1]:
pip install nbformat --break-system-packages -q

In [2]:
!pip install -q pyarrow tqdm

print("Kurulum tamamlandı")

Kurulum tamamlandı


In [3]:
import pandas as pd
import numpy as np
import os, gc, time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm


pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.3f}'.format)

print("Import'lar hazır")

Import'lar hazır


In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'

DOSYALAR = {
    'Video_Games':        DRIVE_PATH + 'amazon_reviews_us_Video_Games_v1_00.tsv',
    'Mobile_Electronics': DRIVE_PATH + 'amazon_reviews_us_Mobile_Electronics_v1_00.tsv',
    'Electronics':        DRIVE_PATH + 'amazon_reviews_us_Electronics_v1_00.tsv',
}

print("Dosya kontrolü:")
print("-" * 55)
for ad, yol in DOSYALAR.items():
    if os.path.exists(yol):
        boyut_mb = os.path.getsize(yol) / 1024**2
        print(f"{ad:<25}  {boyut_mb:>6.0f} MB")
    else:
        print(f"{ad:<25}  BULUNAMADI — yolu kontrol et")

Mounted at /content/drive
Dosya kontrolü:
-------------------------------------------------------
Video_Games                  1149 MB
Mobile_Electronics             56 MB
Electronics                  1646 MB


In [5]:
KULLAN = [
    'product_id', 'product_title', 'product_category',
    'star_rating', 'helpful_votes', 'total_votes',
    'vine', 'verified_purchase', 'review_id',
    'review_headline', 'review_body', 'review_date',
]

TIPLER = {
    'star_rating':       'Int8',
    'helpful_votes':     'Int32',
    'total_votes':       'Int32',
    'vine':              'category',
    'verified_purchase': 'category',
    'product_category':  'category',
}

print("Sütun ayarları hazır")
print(f"Toplam {len(KULLAN)} sütun okunacak (15 sütundan {15-len(KULLAN)}'i atlandı)")

Sütun ayarları hazır
Toplam 12 sütun okunacak (15 sütundan 3'i atlandı)


In [6]:
def dosya_oku(yol, kaynak_adi, chunk_boyutu=100_000):
    boyut_mb = os.path.getsize(yol) / 1024**2
    t0 = time.time()


    if boyut_mb < 200:
        df = pd.read_csv(
            yol,
            sep='\t',
            usecols=KULLAN,
            dtype=TIPLER,
            on_bad_lines='skip',
            low_memory=False,
        )

    else:
        parcalar = []

        reader = pd.read_csv(
            yol, sep='\t', usecols=KULLAN, dtype=TIPLER,
            on_bad_lines='skip', chunksize=chunk_boyutu,
        )

        for parca in tqdm(reader, desc=f"  {kaynak_adi}", unit="chunk"):
            parca = parca[parca['review_body'].notna()]
            parcalar.append(parca)

        df = pd.concat(parcalar, ignore_index=True)
        del parcalar
        gc.collect()
    df['kaynak'] = kaynak_adi

    sure = time.time() - t0
    return df

In [7]:
def dosya_oku(yol, kaynak_adi, chunk_boyutu=100_000):
    boyut_mb = os.path.getsize(yol) / 1024**2
    t0       = time.time()

    SATIR_LIMITI = 100_000

    df = pd.read_csv(
        yol,
        sep='\t',
        usecols=KULLAN,
        dtype=TIPLER,
        on_bad_lines='skip',
        low_memory=False,
        nrows=SATIR_LIMITI,
    )

    df['kaynak'] = kaynak_adi
    print(f"{len(df):,} satır okundu  ({time.time()-t0:.1f}s)")
    return df

In [8]:
print("Dosyalar yükleniyor...")
print("=" * 50)

tum_dosyalar = []
for ad, yol in DOSYALAR.items():
    if not os.path.exists(yol):
        print(f"  ⚠️  {ad} atlandı (dosya yok)")
        continue
    df_parca = dosya_oku(yol, ad)
    tum_dosyalar.append(df_parca)
    gc.collect()

print("\nBirleştiriliyor...")
df_ham = pd.concat(tum_dosyalar, ignore_index=True)
del tum_dosyalar
gc.collect()

print("\n" + "=" * 50)
print(f"Toplam satır   : {len(df_ham):,}")
print(f"Toplam sütun   : {df_ham.shape[1]}")
print(f"RAM kullanımı  : {df_ham.memory_usage(deep=True).sum()/1024**2:.0f} MB")
print("\nKaynak dağılımı:")
print(df_ham['kaynak'].value_counts().to_string())

Dosyalar yükleniyor...
100,000 satır okundu  (2.6s)
100,000 satır okundu  (4.9s)
100,000 satır okundu  (3.4s)

Birleştiriliyor...

Toplam satır   : 300,000
Toplam sütun   : 13
RAM kullanımı  : 239 MB

Kaynak dağılımı:
kaynak
Video_Games           100000
Mobile_Electronics    100000
Electronics           100000


In [9]:
print("İlk 5 satır:")
display(df_ham.head())

print("\nVeri tipleri:")
print(df_ham.dtypes.to_string())

print("\nEksik değer sayıları:")
eksik = df_ham.isnull().sum()
eksik = eksik[eksik > 0]
print(eksik.to_string() if len(eksik) > 0 else "  Eksik değer yok")

print("\nYıldız dağılımı:")
print(df_ham['star_rating'].value_counts().sort_index().to_string())

İlk 5 satır:


,review_id,product_id,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,kaynak
0,RTIS3L2M1F5SM,B001CXYMFS,Thrustmaster T-Flight Hotas X Flight Stick,Video Games,5,0,0,N,Y,an amazing joystick. I especially love that you can twist ...,"Used this for Elite Dangerous on my mac, an amazing joystick. I especially l...",2015-08-31,Video_Games
1,R1ZV7R40OLHKD,B00M920ND6,Tonsee 6 buttons Wireless Optical Silent Gaming Mouse For PC Laptop Gamer Red,Video Games,5,0,0,N,Y,Definitely a silent mouse... Not a single click was heard,"Loved it, I didn't even realise it was a gaming mouse, I typed in &#34;sil...",2015-08-31,Video_Games
2,R3BH071QLH8QMC,B0029CSOD2,Hidden Mysteries: Titanic Secrets of the Fateful Voyage,Video Games,1,0,1,N,Y,One Star,poor quality work and not as it is advertised.,2015-08-31,Video_Games
3,R127K9NTSXA2YH,B00GOOSV98,GelTabz Performance Thumb Grips - PlayStation 4 and PlayStation 3,Video Games,3,0,0,N,Y,"good, but could be bettee","nice, but tend to slip away from stick in intense (hard pressed) gaming sess...",2015-08-31,Video_Games
4,R32ZWUXDJPW27Q,B00Y074JOM,Zero Suit Samus amiibo - Japan Import (Super Smash Bros Series),Video Games,4,0,0,N,Y,Great but flawed.,"Great amiibo, great for collecting. Quality material to be desired, since it...",2015-08-31,Video_Games



Veri tipleri:
review_id              object
product_id             object
product_title          object
product_category       object
star_rating              Int8
helpful_votes           Int32
total_votes             Int32
vine                 category
verified_purchase    category
review_headline        object
review_body            object
review_date            object
kaynak                 object

Eksik değer sayıları:
star_rating           2
helpful_votes         2
total_votes           2
vine                  2
verified_purchase     2
review_headline       4
review_body          53
review_date           2

Yıldız dağılımı:
star_rating
1     38169
2     16568
3     23009
4     45472
5    176780


In [10]:
CIKTI_YOLU = DRIVE_PATH + 'faz1_ham_veri.parquet'

print(f"Kaydediliyor: {CIKTI_YOLU}")
df_ham.to_parquet(CIKTI_YOLU, index=False, compression='snappy')

boyut_mb = os.path.getsize(CIKTI_YOLU) / 1024**2
print(f"Kaydedildi  ({boyut_mb:.0f} MB)")
print(f"\nSonraki adım: Faz2_Temizleme.ipynb'i aç")

Kaydediliyor: /content/drive/MyDrive/faz1_ham_veri.parquet
Kaydedildi  (65 MB)

Sonraki adım: Faz2_Temizleme.ipynb'i aç
